### Imports

In [2]:
from pyspark.ml.recommendation import ALSModel
from pyspark.sql import SparkSession

### Setup

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)
model_path = "results/models/als_25m_base"
model = ALSModel.load(model_path)

df_movies = spark.read.csv("ml-25m/movies.csv",header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/06 22:38:29 WARN Utils: Your hostname, LT-RD-286, resolves to a loopback address: 127.0.1.1; using 192.168.0.234 instead (on interface wlp0s20f3)
26/06/06 22:38:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/06 22:38:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 42104)
Traceback (most recent call last):
  File "/home/regiodata/anaconda3/envs/AIship/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/home/regiodata/anaconda3/envs/AIship/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/home/regiodata/anaconda3/envs/AIship/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/home/regiodata/anaconda3/envs/AIship/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/home/regiodata/anaconda3/envs/AIship/lib/python3.12/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/home/regiodata/anaconda3/envs/AIship/lib/python3.12/site-packages/pyspark/accumulators.

In [9]:

def get_recommendations(model, user_id, num_of_recommendations=10):

    users_df = spark.createDataFrame([(user_id,),], ["userId"])
    recs = model.recommendForUserSubset(users_df, num_of_recommendations)
    movieIds = [rec["movieId"] for rec in recs.collect()[0]["recommendations"]]

    df_movies.where(df_movies.movieId.isin(movieIds)).show()

In [10]:
get_recommendations(model, 1234)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|    858|Godfather, The (1...|         Crime|Drama|
|   2019|Seven Samurai (Sh...|Action|Adventure|...|
|  26082|Harakiri (Seppuku...|               Drama|
|  93040|Civil War, The (1...|     Documentary|War|
| 137904|  I, Claudius (1976)|               Drama|
| 159817| Planet Earth (2006)|         Documentary|
| 171011|Planet Earth II (...|         Documentary|
| 171495|              Cosmos|  (no genres listed)|
| 173351|Wow! A Talking Fi...|Animation|Childre...|
| 179173|Rabbit of Seville...|    Animation|Comedy|
+-------+--------------------+--------------------+



26/06/08 13:05:22 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 416486 ms exceeds timeout 120000 ms
26/06/08 13:05:22 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/08 13:05:27 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$